# Product Ingredient Dataset for FoodEx2 Matching

This notebook builds a dataset from `products_with_language1` using only the `ingredients_tags` column. It expands each product into one row per ingredient and creates an `nlp_text` field suitable for FoodEx2 matching.

In [39]:
import re
import unicodedata
from pathlib import Path

import duckdb
import pandas as pd

DB_PATH = Path("/home/alexl/TFM/openfoodfacts.duckdb")
TAXONOMY_PATH = Path("/home/alexl/TFM/nlp/ingredients.txt")
OUTPUT_PATH = Path("/home/alexl/TFM/data/products_ingredients.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

## Helpers

Parsing, text cleaning, and ingredient validation functions.

- **`clean_text`** — baseline whitespace / punctuation normalizer
- **`clean_ingredient_token`** *(Step D)* — strips trailing digits and embedded percentage quantities
- **`is_valid_ingredient`** *(Step B)* — drops boilerplate text, numeric/unit strings, and tokens that are too short or too long to be ingredients
- **`has_valid_ingredients_tags`** *(Step A)* — drops raw rows whose `ingredients_tags` look like concatenated nutritional tables
- **`normalize_ecode`** *(Step C)* — normalises E-code variants (`e-961`, `E 961`, `e961(i)`) before taxonomy lookup
- **`load_ingredient_taxonomy`** — builds a `synonym → canonical_en` dict from the OFF taxonomy
- **`normalize_with_taxonomy`** — resolves a parsed ingredient to its canonical name, with E-code fallback

In [40]:
def clean_text(text: str) -> str:
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = unicodedata.normalize("NFKC", str(text)).strip()  # collapse fancy Unicode (𝐄𝐠𝐠 → Egg)
    text = re.sub(r"\s+", " ", text)
    text = text.replace("-", " ").replace("_", " ")
    return text.strip()


# ── Step D ────────────────────────────────────────────────────────────────────

_FUNCTIONAL_QUALIFIER_RE = re.compile(
    r"\s+for\s+(colou?r|flavou?r|aroma|texture|preservation|coloring|colouring)\b.*$",
    re.I,
)
_INTERNAL_MEASURE_RE = re.compile(
    r"\b\d+\s*(tbsp|tsp|cups?|tablespoons?|teaspoons?)\b", re.I
)

def clean_ingredient_token(text: str) -> str:
    """Strip quantities, functional qualifiers, and stray leading/trailing digits."""
    text = re.sub(r"\s*\d+[\.,]?\d*\s*%", "", text)       # "cream 35%" → "cream"
    text = re.sub(r"^\d+\s+", "", text)                     # "20 green jalapeño" → "green jalapeño"
    text = _INTERNAL_MEASURE_RE.sub("", text)               # "oil distilled 2 tbsp" → "oil distilled"
    text = _FUNCTIONAL_QUALIFIER_RE.sub("", text)           # "paprika for color" → "paprika"
    text = re.sub(r"(?<=[a-z])\d+$", "", text)              # "juice1" → "juice"
    text = re.sub(r"\s+\d+$", "", text)                     # "green chard 1" → "green chard"
    return text.strip()


# ── Step B ────────────────────────────────────────────────────────────────────

_BOILERPLATE_RE = re.compile(
    r"\b(manufactured|distributed|refrigerat|best before|calorie|calories|"
    r"remove|set aside|made from|plant that|company|consumer information|"
    r"see base|packaging|per serving|daily value|directions|warning|"
    r"product is|for distribution|contains allergen|allergen|"
    r"servings?|flush|consume|responsibly|celebrat|irradiat|processed with|"
    r"use by|sell by|keep refrigerat|store in|"
    r"may also contain|trace of|nutrition|suitable for|less than|per package|contains no)\b",
    re.I,
)
_UNIT_TOKEN_RE = re.compile(
    r"^\d[\d\.,]*\s*(g|mg|kg|ml|l|oz|lb|kcal|kj|cal|iu)?$", re.I
)

# Known valid short English ingredient words (≤5 chars) that must not be dropped
_SHORT_INGREDIENT_WHITELIST = {
    # 3-char
    "oil", "egg", "rye", "oat", "fat", "fig", "yam", "roe", "gin", "rum",
    "soy", "gum", "ice", "pea", "nut",
    # 4-char
    "salt", "milk", "corn", "beef", "pork", "lamb", "rice", "malt", "lard",
    "whey", "wine", "beer", "miso", "tofu", "fish", "lime", "mint", "sage",
    "dill", "clam", "crab", "duck", "agar", "bran", "plum", "date", "oats",
    "eggs", "nuts", "beet", "cane", "palm", "peas", "seed", "tuna", "kale",
    "okra", "figs", "nori", "suet", "veal", "goat", "teff", "kelp", "ghee",
    "cola", "lees", "lard",
    # 5-char
    "sugar", "cream", "water", "wheat", "honey", "yeast", "onion", "lemon",
    "apple", "cocoa", "flour", "spice", "juice", "olive", "thyme", "basil",
    "cacao", "carob", "chili", "clove", "dates", "herbs", "maple", "pasta",
    "spelt", "sumac", "peach", "grape", "melon", "mango", "berry", "prune",
    "agave", "anise", "brine", "caper", "cress", "cumin", "curry", "grain",
    "guava", "kamut", "kombu", "kvass", "leeks", "lupin", "mirin", "natto",
    "okara", "olein", "pecan", "perch", "prawn", "quark", "squid", "stout",
    "syrup", "umami", "vodka", "wafer",
}

def is_valid_ingredient(text: str) -> bool:
    """Return False for tokens that are clearly not ingredients."""
    if len(text) < 3:
        return False
    words = text.split()
    if len(words) > 8:
        return False
    if _BOILERPLATE_RE.search(text):
        return False

    # Non-Latin script: drop if >25% of letter chars are outside Latin range (U+0000–U+024F)
    letters = [c for c in text if unicodedata.category(c)[0] == "L"]
    if letters:
        non_latin = sum(1 for c in letters if ord(c) > 0x024F)
        if non_latin / len(letters) > 0.25:
            return False

    numeric_ratio = sum(1 for w in words if _UNIT_TOKEN_RE.match(w)) / len(words)
    if numeric_ratio > 0.5:
        return False
    digit_ratio = sum(c.isdigit() for c in text) / len(text)
    if digit_ratio > 0.4:
        return False

    # Truncated fragment: drop if ≤2 words and any word is ≤2 chars
    if len(words) <= 2 and any(len(w) <= 2 for w in words):
        return False

    # Single-word short-token filter: drop 1-word tokens of ≤5 chars not in whitelist
    # catches truncated words like "magne", "panto", "cholest" (after next rule)
    if len(words) == 1 and len(text) <= 5 and text.lower() not in _SHORT_INGREDIENT_WHITELIST:
        return False

    return True


# ── Step A ────────────────────────────────────────────────────────────────────

def has_valid_ingredients_tags(tags_str: str) -> bool:
    """Return False when >40% of comma-separated tokens are pure numbers/units."""
    if pd.isna(tags_str) or str(tags_str).strip() == "":
        return False
    tokens = [
        re.sub(r"^[a-z]{2}:", "", t.strip())
        for t in str(tags_str).split(",")
        if t.strip()
    ]
    if not tokens:
        return False
    numeric = sum(1 for t in tokens if _UNIT_TOKEN_RE.match(t))
    return numeric / len(tokens) < 0.4


# ── Step C ────────────────────────────────────────────────────────────────────

_ECODE_RE = re.compile(
    r"^e\s*-?\s*(\d{3,4})\s*([a-z]?\s*(?:\([ivx]+\))?)?$", re.I
)

def normalize_ecode(text: str) -> str:
    """Normalise E-code variants: 'e-961', 'E 322', 'e471(i)' → 'e961', 'e322', 'e471i'."""
    m = _ECODE_RE.match(text.strip())
    if not m:
        return text
    number = m.group(1)
    suffix = re.sub(r"[\s()\-]", "", (m.group(2) or "")).lower()
    return f"e{number}{suffix}"


# ── Taxonomy ──────────────────────────────────────────────────────────────────

def load_ingredient_taxonomy(path: Path) -> dict[str, str]:
    """Parse OFF ingredients.txt into a normalized-synonym → canonical-English dict."""
    taxonomy: dict[str, str] = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.startswith("en:"):
                continue
            parts = [p.strip() for p in line[3:].split(",")]
            if not parts or not parts[0]:
                continue
            canonical = parts[0].lower()
            for synonym in parts:
                synonym = synonym.strip()
                if not synonym:
                    continue
                key = re.sub(
                    r"\s+", " ",
                    synonym.lower().replace("-", " ").replace("_", " ")
                ).strip()
                taxonomy[key] = canonical
    return taxonomy


def normalize_with_taxonomy(ingredient: str, taxonomy: dict[str, str]) -> str:
    """Return the canonical taxonomy name if found, else the original string.

    Falls back to E-code normalisation (Step C) before giving up.
    """
    key = ingredient.lower().strip()
    if key in taxonomy:
        return taxonomy[key]
    normalized = normalize_ecode(key)
    if normalized != key:
        return taxonomy.get(normalized, ingredient)
    return ingredient


# ── Pipeline ──────────────────────────────────────────────────────────────────

def parse_ingredient_tags(tags_str: str) -> list[str]:
    if pd.isna(tags_str) or str(tags_str).strip() == "":
        return []
    ingredients = []
    for raw in str(tags_str).split(","):
        token = raw.strip()
        if not token:
            continue
        token = re.sub(r"^[a-z]{2}:", "", token)
        token = clean_text(token)
        token = clean_ingredient_token(token)   # Step D
        if token:
            ingredients.append(token)
    return ingredients


def build_ingredient_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["product_context"] = df["product_name"].apply(clean_text)
    df["ingredient_list"] = df["ingredients_tags"].apply(parse_ingredient_tags)
    df = df.explode("ingredient_list").reset_index(drop=True)
    df = df[df["ingredient_list"].notna() & (df["ingredient_list"] != "")]
    df = df.rename(columns={"ingredient_list": "ingredient"})
    before = len(df)
    df = df[df["ingredient"].apply(is_valid_ingredient)].reset_index(drop=True)  # Step B
    print(f"  Step B: dropped {before - len(df):,} invalid tokens, {len(df):,} remaining")
    df["nlp_text"] = df["ingredient"]
    return df[["code", "lang_product_name", "product_name", "product_context", "ingredient", "nlp_text"]]

## Load OFF ingredient taxonomy

Parse `ingredients.txt` (the Open Food Facts ingredient taxonomy) into a `synonym → canonical_en` lookup. Keys are normalized the same way `clean_text` works (lowercase, hyphens/underscores → spaces), so they match the parsed ingredient tokens exactly.

In [41]:
taxonomy = load_ingredient_taxonomy(TAXONOMY_PATH)
print(f"Taxonomy loaded: {len(taxonomy):,} entries (canonical names + synonyms)")

Taxonomy loaded: 5,965 entries (canonical names + synonyms)


## Load product data

Load only the columns needed for product–ingredient extraction.

In [42]:
con = duckdb.connect(DB_PATH)
query = """
    SELECT code, lang_product_name, product_name, ingredients_tags
    FROM products_with_language1
    WHERE lang_ingredients_text = 'en'
      AND NOT regexp_matches(
        COALESCE(categories_tags, ''),
        'en:beauty-products|en:cosmetics|en:hygiene-products|'
        'en:pet-food|en:cat-food|en:dog-food|en:animal-food|en:bird-food|'
        'en:cleaning-products|en:detergents|en:non-food-products'
      )
      AND food_groups_tags IS NOT NULL
    USING SAMPLE 10%
"""
df = con.execute(query).fetchdf()

print("Loaded rows:", len(df))
print(df["ingredients_tags"].notna().sum(), "rows with ingredients_tags")
df.head()

Loaded rows: 27412
27404 rows with ingredients_tags


,code,lang_product_name,product_name,ingredients_tags
0,0019646005447,en,Honeycrisp apple juice,en:unfiltered-pasteurized-honeycrisp-apple-jui...
1,0019646005454,en,"Harmons, organic juice, original apple","en:apple,en:fruit,en:malaceous-fruit"
2,0019646005478,en,"Harmons, honeycrisp juice, apple",en:unfiltered-pasteurized-honeycrisp-apple-jui...
3,0019646005669,it,Apple cider vinegar,"en:cider-vinegar,en:vinegar,en:acidity"
4,0019646005812,en,Bronze cut linguine,"en:durum-wheat-semolina,en:cereal,en:wheat,en:..."


## Step A — Pre-expansion row filter

Drop rows whose `ingredients_tags` field is dominated by numeric/unit tokens. These are products where the tags column contains a nutritional table rather than a real ingredient list.

In [43]:
before_a = len(df)
df = df[df["ingredients_tags"].apply(has_valid_ingredients_tags)].reset_index(drop=True)
print(f"Step A: dropped {before_a - len(df):,} rows ({(before_a - len(df)) / before_a:.1%}), {len(df):,} remaining")

Step A: dropped 19 rows (0.1%), 27,393 remaining


## Expand ingredients and create FoodEx2-ready rows

In [44]:
expanded = build_ingredient_rows(df)
print(f"Expanded rows: {len(expanded):,}")
print(f"Unique ingredients: {expanded['ingredient'].nunique():,}")

expanded["nlp_text"] = expanded["ingredient"].apply(lambda x: normalize_with_taxonomy(x, taxonomy))

matched = (expanded["nlp_text"] != expanded["ingredient"]).sum()
print(f"Taxonomy matches: {matched:,} / {len(expanded):,} ({matched / len(expanded):.1%})")
expanded.head(20)

  Step B: dropped 146,866 invalid tokens, 634,573 remaining
Expanded rows: 634,573
Unique ingredients: 25,660
Taxonomy matches: 9,555 / 634,573 (1.5%)


,code,lang_product_name,product_name,product_context,ingredient,nlp_text
0,0019646005454,en,"Harmons, organic juice, original apple","Harmons, organic juice, original apple",apple,apple
1,0019646005454,en,"Harmons, organic juice, original apple","Harmons, organic juice, original apple",malaceous fruit,malaceous fruit
2,0019646005669,it,Apple cider vinegar,Apple cider vinegar,cider vinegar,cider vinegar
3,0019646005669,it,Apple cider vinegar,Apple cider vinegar,vinegar,vinegar
4,0019646005669,it,Apple cider vinegar,Apple cider vinegar,acidity,acidity
5,0019646005812,en,Bronze cut linguine,Bronze cut linguine,durum wheat semolina,durum wheat semolina
6,0019646005812,en,Bronze cut linguine,Bronze cut linguine,cereal,cereal
7,0019646005812,en,Bronze cut linguine,Bronze cut linguine,wheat,wheat
8,0019646005812,en,Bronze cut linguine,Bronze cut linguine,durum wheat,durum wheat
9,0019646005812,en,Bronze cut linguine,Bronze cut linguine,semolina,semolina


## Export dataset

The CSV includes one row per product ingredient, with `nlp_text` ready for matching against FoodEx2 terms.

In [45]:
expanded.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(expanded)} rows to {OUTPUT_PATH}")

Saved 634573 rows to /home/alexl/TFM/data/products_ingredients.csv
